In [ ]:
# Optional: install the tested dependency versions if they are missing.
# Works from the repository root or notebooks/. pickle is part of Python.
from pathlib import Path
_dependency_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / "models" / "requirements.txt").exists())
%pip install -r "{_dependency_root / 'models' / 'requirements.txt'}"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import log_loss
from sklearn.metrics import accuracy_score
import pickle

In [ ]:
from pathlib import Path

# Works when launched from either the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "processed dataset" / "X_processed.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the repository or its notebooks directory.")
X_processed = pd.read_csv(ROOT / "processed dataset" / "X_processed.csv", index_col=0)
y_frame = pd.read_csv(ROOT / "processed dataset" / "y_processed.csv", index_col=0)
if y_frame.shape[1] != 1 or not X_processed.index.equals(y_frame.index):
    raise ValueError("Expected one target column with the same row index as X.")
y_processed = y_frame.iloc[:, 0]
if set(y_processed.unique()) != {0, 1}:
    raise ValueError("Expected a binary 0/1 target with no missing values.")

In [ ]:
# Baseline B retains every processed feature, including demographics.
excluded_columns = []
if X_processed.isna().any().any() or not all(pd.api.types.is_numeric_dtype(t) for t in X_processed.dtypes):
    raise ValueError("Features must be numeric with no missing values.")
if not np.isfinite(X_processed.to_numpy()).all():
    raise ValueError("Features must contain finite numeric values.")
print(f"Baseline B retains all {X_processed.shape[1]} processed features.")

### Splitting the dataset

Baseline B retains gender, ethnicity, and region features that Baseline A excludes: 51 processed features versus 30. The stratified 80/20 split and seed match A's saved artifact so the same rows are held out. Demographic inclusion does not establish fairness; subgroup evaluation remains necessary.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed
)
print(f"Training: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print("Training class counts:", y_train.value_counts().to_dict())

In [ ]:
def train_test_LR(X_train, X_test, y_train, y_test, iterations, solve , weight = 'balanced', c = 1.0):
    # Create a logistic regression model
    model = LogisticRegression(class_weight= weight, C=c, max_iter = iterations, solver = solve)

    # Fit the model to the training data
    model.fit(X_train, y_train)

    #Predict the probability of the class of examples of the test data

    proba_predict = model.predict_proba(X_test)

    #Computing the log loss of the probability (proba_predict) by comparing 
    #the actual labels of the test data to the probability predictions

    logloss = log_loss(y_test, proba_predict)

    #Making predictions on the test data, to evaluate the accuracy score
    
    class_label_predict = model.predict(X_test)

    #Computing the accuracy score of the model by comparing the y_test
    #with the class label predictions generated from the step above
    
    accscore = accuracy_score(y_test, class_label_predict)
    
    return {"model": model, "predicted_proba": proba_predict, 'accuracy_score': accscore, 'log_loss': logloss, "predicted_classes": class_label_predict}

#### Pickling the Model 

In [ ]:
def pickling(model, model_name):
    with open(model_name, "wb") as handle:
        pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)
    # Only load pickle files produced here or from another trusted source.
    with open(model_name, "rb") as handle:
        return pickle.load(handle)

### Automated hyperparameter optimization

GridSearchCV evaluates **200 configurations** (100 regularization strengths times 2 class-weight choices), with five stratified folds: **1,000 validation fits**, followed by refitting the winner on the training split. Smaller `C` means stronger regularization. L2 regularization is fixed; `newton-cholesky` is suitable for many rows and relatively few features. This matches the search that produced A's saved model artifact.

The selection metric is **average precision**, which summarizes precision and recall across thresholds for the minority adverse-outcome class. The test set is reserved for final evaluation and never selects hyperparameters. Scaling is fitted inside each fold through a pipeline. The upstream `Healthcare1C.ipynb` already fits scaling before exporting the full dataset, which limits validation estimates. Definitive validation should split raw data before fitting upstream preprocessing. Patient identifiers are absent from these processed files, preventing patient-group separation here.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, balanced_accuracy_score

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(solver="newton-cholesky", max_iter=1000, random_state=42)),
])
param_grid = {
    "logistic__C": np.logspace(-4, 3, 100),
    "logistic__class_weight": [None, "balanced"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipeline, param_grid, scoring={"average_precision": "average_precision", "roc_auc": "roc_auc"},
    refit="average_precision", cv=cv, n_jobs=4, pre_dispatch=4,
    return_train_score=False, error_score="raise", verbose=1,
)
print(f"Searching {len(list(ParameterGrid(param_grid)))} configurations.")
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best parameters:", search.best_params_)
print(f"Best mean CV average precision: {search.best_score_:.6f}")
cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_average_precision")
display(cv_results[["params", "mean_test_average_precision", "std_test_average_precision", "mean_test_roc_auc"]].head(10))

### Final held-out evaluation

In [ ]:
test_proba = best_model.predict_proba(X_test)[:, list(best_model.classes_).index(1)]
test_predictions = best_model.predict(X_test)
test_metrics = {
    "average_precision": float(average_precision_score(y_test, test_proba)),
    "roc_auc": float(roc_auc_score(y_test, test_proba)),
    "accuracy": float(accuracy_score(y_test, test_predictions)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, test_predictions)),
    "adverse_recall_at_0_5": float((test_predictions[y_test.to_numpy() == 1] == 1).mean()),
    "predicted_adverse_count_at_0_5": int((test_predictions == 1).sum()),
    "log_loss": float(log_loss(y_test, test_proba)),
}
print(pd.Series(test_metrics).to_string())
print(classification_report(y_test, test_predictions, zero_division=0))
print("Confusion matrix (rows=true, columns=predicted; classes 0, 1):")
print(confusion_matrix(y_test, test_predictions, labels=[0, 1]))

### Export and verify the best model

The `.pkl` stores the fitted scaler and logistic regression together. It expects all 51 **processed numeric features**, including demographics, in the recorded order; it does not transform raw patient records. Export the training-split winner so the held-out evaluation describes this exact artifact. Only aggregate metrics and feature names are saved alongside it. Baseline B filenames keep A's artifacts intact.

In [ ]:
import json
import platform
import sklearn

MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
model_path = MODEL_DIR / "logistic_regression_baseline_b_best.pkl"
loaded_model = pickling(best_model, model_path)
np.testing.assert_allclose(loaded_model.predict_proba(X_test), best_model.predict_proba(X_test))
metadata = {
    "baseline": "B",
    "fixed_model_parameters": {"solver": "newton-cholesky", "regularization": "L2", "max_iter": 1000},
    "target": y_processed.name,
    "positive_class": 1,
    "feature_columns": list(X_train.columns),
    "excluded_columns": excluded_columns,
    "input_contract": "Already processed numeric features in feature_columns order; raw records are not supported.",
    "selection_metric": "average_precision",
    "best_parameters": search.best_params_,
    "best_cv_average_precision": float(search.best_score_),
    "test_metrics": test_metrics,
    "candidate_count": len(search.cv_results_["params"]),
    "cv_folds": 5,
    "training_rows": len(X_train),
    "test_rows": len(X_test),
    "random_state": 42,
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
}
(MODEL_DIR / "logistic_regression_baseline_b_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
cv_results.to_csv(MODEL_DIR / "logistic_regression_baseline_b_cv_results.csv", index=False)
print(f"Saved and reload-verified: {model_path.relative_to(ROOT)}")

### Summary

**Feature Selection**: Retained all 51 processed features, including gender, ethnicity, and region. Baseline A excludes these demographic features and uses 30. Checked feature/label alignment, finite numeric inputs, and a one-dimensional binary target.

**Automated Optimization**: Added a reproducible five-fold stratified GridSearchCV covering 200 combinations of L2 regularization strength and class weighting. Scaling is fitted inside each training fold. The search and 80/20 split match those used for A's saved model artifact.

**Model Selection and Evaluation**: Selected the highest mean cross-validation average precision, then evaluated the winner once on the reserved test set. Reported average precision, ROC AUC, accuracy, balanced accuracy, log loss, adverse-class recall, the classification report, and confusion matrix. Best means best among this search's candidates for the selection metric; improvement over A is not guaranteed.

**Model Persistence**: Exported the fitted pipeline to `models/logistic_regression_baseline_b_best.pkl`, verified all test-set probabilities after reload, and saved feature order, selected settings, software versions, aggregate metrics, and all search results in B-specific files.

**Sharing the Deliverable**: The `.pkl`, metadata, and results are local repository artifacts. Teammates can load the trusted pipeline without retraining, using matching dependency versions and processed input features. Commit and push the files through the normal Git workflow to upload them to GitHub; saving does not upload automatically. Follow the dataset's terms for sharing derived models.

**Limitations**: The upstream preprocessing fits scaling on the full dataset, and patient identifiers are unavailable for grouped validation. Probability ranking does not guarantee useful detection at the default 0.5 threshold; use the reported adverse recall and confusion matrix to assess that cutoff. Choose any alternative cutoff using training-only validation predictions and evaluate on fresh held-out data. Including demographic features does not establish fairness. This is a research baseline, not a clinically validated allocation system.

In [ ]:
# Example: load a trusted artifact and score already-processed records.
# Match the package versions in the metadata when sharing the model.
with open(MODEL_DIR / "logistic_regression_baseline_b_metadata.json", encoding="utf-8") as handle:
    saved_metadata = json.load(handle)
with open(model_path, "rb") as handle:
    shared_model = pickle.load(handle)
example_probabilities = shared_model.predict_proba(
    X_test.loc[:, saved_metadata["feature_columns"]].head(5)
)
print("Reloaded pipeline successfully; probability shape:", example_probabilities.shape)